<h1 style="color:DodgerBlue">Итоговый проект </h1>
<h2 style="color:DeepSkyBlue"> по дисциплине «Объектно-ориентированное программирование C#»</h2>

----

<h2 style="color:DodgerBlue">Название проекта: "Магазин одежды"</h2>


### Вариант задания 8






----
<h2 style="color:DodgerBlue">Описание проекта: Онлайн-магазин для продажи одежды.</h2>


# Техническое задание:
#####	Разработать классы для товаров, категорий и заказов.
#####	Создать интерфейс для поиска и добавления товаров в корзину.
#####	Включить обработку и историю заказов.
#####	Использовать Blazor в качестве Web-интерфейса и для реализации пользовательского взаимодействия.


In [17]:
//✅// Базовый интерфейс для всех сущностей                1. БАЗОВЫЕ КЛАССЫ И ИНТЕРФЕЙСЫ        ✅
public interface IEntity//Сущность
{
    int Id { get; set; }
    string Name { get; set; }
}

// Интерфейс для товаров
public interface IProduct : IEntity//Product
{
    decimal Price { get; set; }     //Цена
    string Description { get; set; }//Описание
    string ImageUrl { get; set; }   //URL-адрес изображения
    bool IsAvailable { get; set; }  //Доступно
}

// Интерфейс для корзины
public interface IShoppingCart
{
    void AddItem(Product product, int quantity);
    void RemoveItem(int productId);
    decimal CalculateTotal();
    void Clear();
}

// Интерфейс для заказов
public interface IOrder
{
    void ProcessOrder();//Процесс заказа
    void CancelOrder();//Отменить заказ
    string GetOrderStatus();
}



//✅//Базовые классы:                                                                             ✅
public abstract class BaseEntity : IEntity
{
    public int Id { get; set; }
    public string Name { get; set; } = string.Empty;
    public DateTime CreatedAt { get; set; } = DateTime.Now;//Создано в
    public DateTime? UpdatedAt { get; set; }
}

// Абстрактный класс для товаров
public abstract class Product : BaseEntity, IProduct
{
    public decimal Price { get; set; }                          //Цена
    public string Description { get; set; } = string.Empty;     //Описание
    public string ImageUrl { get; set; } = string.Empty;        //URL-адрес изображения
    public bool IsAvailable { get; set; } = true;               //Доступно
    public int StockQuantity { get; set; }                      //Количество
    
    //  ДЕЛЕГАТ для уведомлений
    public delegate void ProductEventHandler(string message, Product product);//Обработчик событий продукта
    
    //  СОБЫТИЯ
    public event ProductEventHandler OnPriceChanged;//Цена изменилась
    public event ProductEventHandler OnStockChanged;//На складе изменено
    
    public virtual void UpdatePrice(decimal newPrice)
    {
        var oldPrice = Price;
        Price = newPrice;
        OnPriceChanged?.Invoke($"Цена изменена с {oldPrice} на {newPrice}", this);
    }
    
    public virtual void UpdateStock(int newQuantity)
    {
        StockQuantity = newQuantity;
        OnStockChanged?.Invoke($"Количество на складе изменено на {newQuantity}", this);
    }
}



//✅//Одежда с размерами:                                  2. КЛАССЫ ДЛЯ ОДЕЖДЫ (НАСЛЕДОВАНИЕ)    ✅
public class ClothingProduct : Product
{
    public string Size { get; set; } = string.Empty; // S, M, L, XL
    public string Color { get; set; } = string.Empty;
    public string Material { get; set; } = string.Empty;
    public string Brand { get; set; } = string.Empty;
    public string Category { get; set; } = string.Empty; // Мужская, Женская, Детская
    
    // ПЕРЕГРУЗКА методов
    public void UpdateSize(string newSize)
    {
        Size = newSize;
        Console.WriteLine($"Размер изменен на: {newSize}");
    }
    
    public void UpdateSize(string newSize, bool updateInventory)
    {
        Size = newSize;
        if (updateInventory)
        {
            UpdateStock(StockQuantity + 10); // Пример логики
        }
        Console.WriteLine($"Размер изменен на: {newSize}, инвентарь обновлен");
    }
    
    public override string ToString() //В строку 
    {
        return $"{Name} - {Brand} - {Size} - {Price:C}";
    }
}




//✅//Специализированные классы одежды:                                                           ✅  
//  ПРОСТОЕ НАСЛЕДОВАНИЕ
public class TShirt : ClothingProduct
{
    public string NeckType { get; set; } = string.Empty; // V-образный, круглый     Тип шеи
    public bool HasPrint { get; set; }
    
    public void ApplyDiscount(decimal discountPercent)//Применить скидку
    {
        var discountedPrice = Price * (1 - discountPercent / 100);
        UpdatePrice(discountedPrice);
        Console.WriteLine($"Применена скидка {discountPercent}%");
    }
}

//  СЛОЖНОЕ НАСЛЕДОВАНИЕ
public class Jeans : ClothingProduct
{
    public string Fit { get; set; } = string.Empty; // Slim, Regular, Relaxed       Посадка // Стройная, обычная, свободная
    public string Wash { get; set; } = string.Empty; // Dark, Light, Distressed     Стирка // Темная, светлая, потертая
    
    public override void UpdatePrice(decimal newPrice)
    {
        base.UpdatePrice(newPrice);
        Console.WriteLine($"Цена джинсов обновлена: {newPrice:C}");
    }
}

public class DesignerJeans : Jeans //дизайнерские джинсы
{
    public string Designer { get; set; } = string.Empty;
    public bool IsLimitedEdition { get; set; }
    
    public void MarkAsLimited()//Отметить как ограниченное
    {
        IsLimitedEdition = true;
        UpdatePrice(Price * 1.5m); // Наценка за лимитированную серию
        Console.WriteLine($"Джинсы помечены как лимитированная серия от {Designer}");
    }
}



//✅//Корзина покупок:                                      3. КОРЗИНА И ЗАКАЗЫ                   ✅
public class ShoppingCart : IShoppingCart
{
    //  КОЛЛЕКЦИЯ для хранения items
    private List<CartItem> _items = new List<CartItem>();
    
    public string CartId { get; private set; }
    public DateTime CreatedAt { get; private set; }
    
    //  ДЕЛЕГАТЫ и СОБЫТИЯ
    public delegate void CartEventHandler(string message, ShoppingCart cart);  //Обработчик событий корзины
    public event CartEventHandler OnCartUpdated;
    
    public ShoppingCart()
    {
        CartId = Guid.NewGuid().ToString();
        CreatedAt = DateTime.Now;
    }
    
    public void AddItem(Product product, int quantity = 1)
    {
        var existingItem = _items.FirstOrDefault(i => i.Product.Id == product.Id);
        
        if (existingItem != null)
        {
            existingItem.Quantity += quantity;
        }
        else
        {
            _items.Add(new CartItem { Product = product, Quantity = quantity });
        }
        
        OnCartUpdated?.Invoke($"Добавлен товар: {product.Name}", this);
    }
    
    public void RemoveItem(int productId)
    {
        var item = _items.FirstOrDefault(i => i.Product.Id == productId);
        if (item != null)
        {
            _items.Remove(item);
            OnCartUpdated?.Invoke($"Удален товар: {item.Product.Name}", this);
        }
    }
    
    public decimal CalculateTotal()
    {
        return _items.Sum(item => item.Product.Price * item.Quantity);
    }
    
    public void Clear()
    {
        _items.Clear();
        OnCartUpdated?.Invoke("Корзина очищена", this);
    }
    
    //  LINQ для работы с коллекцией   технология платформы .NET
    public IEnumerable<CartItem> GetItemsByCategory(string category)
    {
        return _items.Where(item => item.Product is ClothingProduct clothing && 
                                   clothing.Category == category);
    }
    
    public int GetTotalItems() => _items.Sum(item => item.Quantity);
    
    public void DisplayCartContents()
    {
        Console.WriteLine($"Корзина {CartId}:");
        foreach (var item in _items)
        {
            Console.WriteLine($"  {item.Product.Name} x {item.Quantity} - {item.Product.Price * item.Quantity:C}");
        }
        Console.WriteLine($"Итого: {CalculateTotal():C}");
    }
}

public class CartItem
{
    public Product Product { get; set; } = null!;
    public int Quantity { get; set; } = 1;
    
    public decimal TotalPrice => Product.Price * Quantity;
}



//✅//Система заказов:                                                                            ✅
public class Order : BaseEntity, IOrder     // Заказ: Базовая сущность, Заказ
{
    //  КОЛЛЕКЦИЯ для items заказа
    public List<OrderItem> Items { get; private set; } = new List<OrderItem>();
    public string CustomerName { get; set; } = string.Empty;
    public string CustomerEmail { get; set; } = string.Empty;
    public string ShippingAddress { get; set; } = string.Empty;
    public decimal TotalAmount { get; private set; }
    public OrderStatus Status { get; private set; } = OrderStatus.Pending;
    
    //  GENERIC класс для работы с историей
    private OrderHistoryTracker<OrderStatus> _statusHistory = new OrderHistoryTracker<OrderStatus>();
    
    public Order(string customerName, string customerEmail, string shippingAddress)
    {
        CustomerName = customerName;
        CustomerEmail = customerEmail;
        ShippingAddress = shippingAddress;
        _statusHistory.AddEntry(OrderStatus.Pending, "Заказ создан");
    }
    
    public void AddItem(Product product, int quantity)
    {
        Items.Add(new OrderItem { Product = product, Quantity = quantity });
        TotalAmount = Items.Sum(item => item.TotalPrice);
    }
    
    public void ProcessOrder()
    {
        Status = OrderStatus.Processing;
        _statusHistory.AddEntry(Status, "Заказ в обработке");
        Console.WriteLine($"Заказ {Id} обрабатывается");
    }
    
    public void CancelOrder()
    {
        Status = OrderStatus.Cancelled;
        _statusHistory.AddEntry(Status, "Заказ отменен");
        Console.WriteLine($"Заказ {Id} отменен");
    }
    
    public string GetOrderStatus() => Status.ToString();
    
    public void DisplayOrderHistory()
    {
        Console.WriteLine($"История заказа {Id}:");
        foreach (var entry in _statusHistory.GetHistory())
        {
            Console.WriteLine($"  {entry.Timestamp}: {entry.Value} - {entry.Notes}");
        }
    }
}

public class OrderItem
{
    public Product Product { get; set; } = null!;
    public int Quantity { get; set; } = 1;
    public decimal TotalPrice => Product.Price * Quantity;
}

public enum OrderStatus
{
    Pending,
    Processing,
    Shipped,
    Delivered,
    Cancelled
}



//✅//Generic репозиторий:           4. GENERIC КЛАССЫ И СЕРВИСЫ                                  ✅
//  GENERIC класс для работы с коллекциями
public class Repository<T> where T : BaseEntity
{
    protected List<T> _items = new List<T>();
    private int _nextId = 1;
    
    public delegate void RepositoryEventHandler(string message, T item);
    public event RepositoryEventHandler OnItemAdded;
    public event RepositoryEventHandler OnItemUpdated;
    public event RepositoryEventHandler OnItemDeleted;
    
    public void Add(T item)
    {
        item.Id = _nextId++;
        _items.Add(item);
        OnItemAdded?.Invoke($"Добавлен: {item.Name}", item);
    }
    
    public T GetById(int id) => _items.FirstOrDefault(item => item.Id == id);
    
    public IEnumerable<T> GetAll() => _items.AsReadOnly();
    
    public IEnumerable<T> Find(Func<T, bool> predicate) => _items.Where(predicate);
    
    public void Update(T item)
    {
        var existingItem = GetById(item.Id);
        if (existingItem != null)
        {
            _items.Remove(existingItem);
            _items.Add(item);
            OnItemUpdated?.Invoke($"Обновлен: {item.Name}", item);
        }
    }
    
    public void Delete(int id)
    {
        var item = GetById(id);
        if (item != null)
        {
            _items.Remove(item);
            OnItemDeleted?.Invoke($"Удален: {item.Name}", item);
        }
    }
}

//  Generic класс для истории
public class OrderHistoryTracker<T>
{
    private List<HistoryEntry<T>> _history = new List<HistoryEntry<T>>();
    
    public void AddEntry(T value, string notes)
    {
        _history.Add(new HistoryEntry<T> 
        { 
            Value = value, 
            Notes = notes, 
            Timestamp = DateTime.Now 
        });
    }
    
    public IEnumerable<HistoryEntry<T>> GetHistory() => _history.AsReadOnly();
    
    public HistoryEntry<T> GetLatest() => _history.LastOrDefault();
}

public class HistoryEntry<T>
{
    public T Value { get; set; } = default!;
    public string Notes { get; set; } = string.Empty;
    public DateTime Timestamp { get; set; }
}



//✅//Сервисы:                                                                                    ✅
//  Dependency Injection    Внедрение зависимостей
public interface IProductService
{
    IEnumerable<Product> GetProducts();
    IEnumerable<Product> SearchProducts(string searchTerm);
    Product GetProductById(int id);
    void AddProduct(Product product);
}

public class ProductService : IProductService
{
    private readonly Repository<Product> _productRepository;
    
    public ProductService(Repository<Product> productRepository)
    {
        _productRepository = productRepository;
    }
    
    public IEnumerable<Product> GetProducts() => _productRepository.GetAll();
    
    public IEnumerable<Product> SearchProducts(string searchTerm)
    {
        return _productRepository.Find(p => 
            p.Name.Contains(searchTerm, StringComparison.OrdinalIgnoreCase) ||
            p.Description.Contains(searchTerm, StringComparison.OrdinalIgnoreCase));
    }
    
    public Product GetProductById(int id) => _productRepository.GetById(id);
    
    public void AddProduct(Product product) => _productRepository.Add(product);
}





Console.WriteLine("=== ТЕСТИРОВАНИЕ СИСТЕМЫ МАГАЗИНА ОДЕЖДЫ ===\n");
        
        // Создаем продукты
        var tshirt = new TShirt 
        { 
            Name = "Футболка Basic", 
            Price = 1500, 
            Size = "M", 
            Color = "Черный",
            Brand = "Nike"
        };
        
        var jeans = new Jeans
        {
            Name = "Джинсы Slim Fit",
            Price = 3500,
            Size = "32",
            Color = "Синий",
            Brand = "Levi's"
        };
        
        // Тестируем корзину
        var cart = new ShoppingCart();
        cart.OnCartUpdated += (message, cart) => 
            Console.WriteLine($" {message}");
            
        cart.AddItem(tshirt, 2);
        cart.AddItem(jeans);
        
        // Тестируем события
        tshirt.OnPriceChanged += (message, product) =>
            Console.WriteLine($" {message}");
            
        tshirt.UpdatePrice(1200); // Скидка
        
        Console.WriteLine($"\n Итого в корзине: {cart.CalculateTotal():C}");

=== ТЕСТИРОВАНИЕ СИСТЕМЫ МАГАЗИНА ОДЕЖДЫ ===

 Добавлен товар: Футболка Basic
 Добавлен товар: Джинсы Slim Fit
 Цена изменена с 1500 на 1200

 Итого в корзине: 3 600,00 ¤
